In [3]:
import os
import duckdb
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [4]:
# Load all processed text files
text_dir = "../data/processed/sec_text"

documents = []

for filename in os.listdir(text_dir):
    if filename.endswith('.txt'):
        filepath = os.path.join(text_dir, filename)
        
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Extract ticker from filename e.g. AAPL_2024-01-01.txt
        ticker = filename.split('_')[0]
        filing_date = filename.split('_')[1].replace('.txt', '')
        
        # Create LangChain Document
        doc = Document(
            page_content=content,
            metadata={
                'ticker': ticker,
                'filing_date': filing_date,
                'source': filename
            }
        )
        documents.append(doc)
        print(f"✅ Loaded {filename}")

print(f"\nTotal documents loaded: {len(documents)}")

✅ Loaded AAPL_0000320193-23-000106.txt
✅ Loaded AAPL_0000320193-24-000123.txt
✅ Loaded AAPL_0000320193-25-000079.txt
✅ Loaded AMZN_0001018724-24-000008.txt
✅ Loaded AMZN_0001018724-25-000004.txt
✅ Loaded AMZN_0001018724-26-000004.txt
✅ Loaded BAC_0000070858-24-000122.txt
✅ Loaded BAC_0000070858-25-000139.txt
✅ Loaded BAC_0000070858-26-000157.txt
✅ Loaded GS_0000886982-24-000006.txt
✅ Loaded GS_0000886982-25-000005.txt
✅ Loaded GS_0000886982-26-000091.txt
✅ Loaded JNJ_0000200406-24-000013.txt
✅ Loaded JNJ_0000200406-25-000038.txt
✅ Loaded JNJ_0000200406-26-000016.txt
✅ Loaded JPM_0000019617-24-000225.txt
✅ Loaded JPM_0000019617-25-000270.txt
✅ Loaded JPM_0001628280-26-008131.txt
✅ Loaded MSFT_0000950170-23-035122.txt
✅ Loaded MSFT_0000950170-24-087843.txt
✅ Loaded MSFT_0000950170-25-100235.txt
✅ Loaded TSLA_0001628280-24-002390.txt
✅ Loaded TSLA_0001628280-25-003063.txt
✅ Loaded TSLA_0001628280-26-003952.txt
✅ Loaded WMT_0000104169-24-000056.txt
✅ Loaded WMT_0000104169-25-000021.txt
✅ L

In [5]:
# Split into smaller chunks for better retrieval
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # 1000 chars per chunk
    chunk_overlap=200,    # 200 char overlap between chunks
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0].page_content[:300])
print(f"\nMetadata: {chunks[0].metadata}")

Total chunks created: 1982

Sample chunk:
Company: AAPL
Filing Date: 0000320193-23-000106

Metadata: {'ticker': 'AAPL', 'filing_date': '0000320193-23-000106', 'source': 'AAPL_0000320193-23-000106.txt'}


In [6]:
# Load free local embedding model
print("Loading embedding model... (first time may take 2-3 minutes to download)")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",  # small, fast, free
    model_kwargs={'device': 'cpu'}
)

print("✅ Embedding model loaded!")

# Create ChromaDB vector store
chroma_path = "../data/chroma_db"

print("Creating vector store... this may take 5-10 minutes")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=chroma_path
)

print(f"✅ Vector store created with {vectorstore._collection.count()} chunks!")

Loading embedding model... (first time may take 2-3 minutes to download)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 5306.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded!
Creating vector store... this may take 5-10 minutes
✅ Vector store created with 1982 chunks!


In [7]:
# Create retriever - fetches top 5 most relevant chunks per query
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Test retrieval
test_query = "What are Apple's main risk factors?"
results = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(results)} chunks\n")

for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} [{doc.metadata['ticker']} | {doc.metadata['filing_date']}] ---")
    print(doc.page_content[:200])
    print()

Query: What are Apple's main risk factors?
Retrieved 5 chunks

--- Chunk 1 [AAPL | 0000320193-24-000123] ---
. In addition to an adverse impact on demand for the Company&#8217;s products and services, uncertainty about, or decline in, global or regional economic conditions can have significant impact on the 

--- Chunk 2 [AAPL | 0000320193-25-000079] ---
. Potential outcomes include financial instability; inability to obtain credit to finance business operations; and insolvency. Adverse economic conditions can also lead to increased credit and collect

--- Chunk 3 [AAPL | 0000320193-23-000106] ---
. These and other impacts can materially adversely affect the Company&#8217;s business, results of operations, financial condition and stock price. The Company&#8217;s business can be impacted by poli

--- Chunk 4 [AAPL | 0000320193-24-000123] ---
. Apple Inc. 2024 Form 10-K The Company&#8217;s ability to compete successfully depends heavily on ensuring the continuing and timely introduction 

In [10]:
from langchain_core.prompts import PromptTemplate
import warnings
warnings.filterwarnings('ignore')

# Custom prompt template
prompt_template = """You are a financial risk analyst assistant.
Use the following SEC filing excerpts to answer the question.
Always mention which company and filing date the information comes from.
If you don't know, say "I don't have enough information."

Context from SEC Filings:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("✅ Prompt template ready!")

✅ Prompt template ready!


In [12]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0.1,
    max_tokens=1024,
    api_key=os.getenv("GROQ_API_KEY"),
)

# Modern LangChain approach using LCEL (LangChain Expression Language)
def format_docs(docs):
    return "\n\n".join([
        f"[{doc.metadata['ticker']} | {doc.metadata['filing_date']}]\n{doc.page_content}"
        for doc in docs
    ])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | PROMPT
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready with Groq (llama3-8b-8192)!")


✅ RAG chain ready with Ollama!


In [13]:
def ask_copilot(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Get answer
    answer = rag_chain.invoke(question)
    print(f"Answer:\n{answer}")
    
    # Get sources separately
    print("\nSources used:")
    docs = retriever.invoke(question)
    for doc in docs:
        print(f"  📄 {doc.metadata['ticker']} | {doc.metadata['filing_date']}")

# Test it!
ask_copilot("What are Tesla's main risk factors?")

Question: What are Tesla's main risk factors?
--------------------------------------------------
Answer:
 Based on the provided SEC filings, Tesla's main risk factors include:

1. The events and consequences discussed in these risk factors could have a material adverse effect on Tesla's business, growth, reputation, prospects, financial condition, operating results, cash flows, liquidity, and stock price ([TSLA | 0001628280-24-002390]).
2. The risks associated with property, plant, and equipment, net leased assets, long-term debt, and finance leases, both current and noncurrent ([TSLA | 0001628280-24-002390] and [TSLA | 0001628280-23-1231]).
3. The risks associated with digital assets net noncurrent ([TSLA | 0001628280-23-1231]).
4. The design, development, manufacture, sale, and lease of high-performance fully electric vehicles and energy generation and storage systems, and the related services, could be subject to various risks ([TSLA | 0001628280-24-002390]).
5. The widespread adopt

In [14]:
ask_copilot("How does JPMorgan describe its credit risk management?")

Question: How does JPMorgan describe its credit risk management?
--------------------------------------------------
Answer:
 I don't have enough information as the provided SEC filings are for Apple (AAPL) and Tesla (TSLA), not JPMorgan.

Sources used:
  📄 AAPL | 0000320193-25-000079
  📄 AAPL | 0000320193-24-000123
  📄 AAPL | 0000320193-23-000106
  📄 TSLA | 0001628280-25-003063
  📄 TSLA | 0001628280-24-002390


In [15]:
ask_copilot("Which company mentions inflation as a risk factor?")

Question: Which company mentions inflation as a risk factor?
--------------------------------------------------
Answer:
 I don't have enough information to determine if either Amazon (AMZN) or Apple (AAPL), based on the provided SEC filings, mentions inflation as a risk factor. The excerpts given do not explicitly mention inflation, but they discuss various risks that could materially adversely affect their business operations and financial condition, including global and regional economic conditions, which can indirectly be related to inflation. To confirm if inflation is explicitly mentioned, further examination of the full filings would be necessary.

Sources used:
  📄 AMZN | 0001018724-25-000004
  📄 AAPL | 0000320193-24-000123
  📄 AMZN | 0001018724-24-000008
  📄 AAPL | 0000320193-24-000123
  📄 AAPL | 0000320193-23-000106
